# repr-audit Demo

This notebook demonstrates the full `repr-audit` workflow:
1. Audit a single model
2. Compare multiple models
3. Analyse polysemous tokens
4. Export and load results

In [ ]:
# Install if needed
 !pip install repr-audit

IndentationError: unexpected indent (1339211375.py, line 2)

In [5]:
import numpy as np
from repr_audit import RepresentationAuditor
from repr_audit.plotting import plot_model_comparison

ModuleNotFoundError: No module named 'repr_audit'

## 1. Sentences

We use a small set of sentences with deliberately polysemous words
("bank", "bat", "spring") to exercise the self-similarity metric.

In [ ]:
sentences = [
    # Financial bank
    "She deposited her savings at the bank.",
    "The bank approved the mortgage application.",
    "Interest rates at the bank rose sharply.",
    "The central bank announced new policies.",
    # River bank
    "We picnicked on the grassy bank of the river.",
    "The riverbank was eroded by the flood.",
    "Flowers lined the bank near the water.",
    "Fish could be seen jumping near the muddy bank.",
    # General context sentences
    "The scientist published her findings in a journal.",
    "Machine learning models require large datasets.",
    "The concert was held in an outdoor amphitheatre.",
    "He decided to change careers after ten years.",
    "Renewable energy is critical for the future.",
    "The restaurant opened a new branch downtown.",
    "She learned to play the piano as a child.",
    "The committee approved the annual budget.",
]

# Polysemous token contexts for self-similarity
token_sentences = {
    "bank": [
        "The bank charged a high fee.",
        "She sat on the bank of the river.",
        "He robbed the bank at noon.",
        "The bank of clouds moved slowly.",
        "We need to bank the fire for the night.",
    ]
}

# Labels: 0 = financial sentence, 1 = river sentence, 2 = other
labels = np.array([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2])

## 2. Audit BERT

In [ ]:
auditor_bert = RepresentationAuditor("bert-base-uncased")

results_bert = auditor_bert.audit(
    sentences=sentences,
    token_sentences=token_sentences,
    labels=labels,
)

print(results_bert.summary())

In [ ]:
results_bert.plot()

## 3. Compare BERT, RoBERTa, GPT-2

In [ ]:
models = [
    "bert-base-uncased",
    "roberta-base",
    "gpt2",
]

all_results = []
for model_name in models:
    auditor = RepresentationAuditor(model_name)
    r = auditor.audit(sentences, labels=labels)
    all_results.append(r)
    print(f"\n--- {model_name} ---")
    print(r.summary())

In [ ]:
plot_model_comparison(all_results, metric="anisotropy")

In [ ]:
plot_model_comparison(all_results, metric="mev")

## 4. Results Table (for README / paper)

This produces the table you paste into your README.

In [ ]:
import pandas as pd

rows = []
for r in all_results:
    rows.append({
        "Model": r.model_name,
        "Avg Anisotropy": f"{r.avg_anisotropy:.4f}",
        "Min Anisotropy Layer": r.min_anisotropy_layer,
        "Semantic Onset Depth": r.semantic_onset_depth,
        "Max MEV Layer": r.max_mev_layer,
    })

df = pd.DataFrame(rows)
print(df.to_markdown(index=False))

## 5. Save and Load Results

In [ ]:
from repr_audit import AuditResults

results_bert.to_json("bert_results.json")
loaded = AuditResults.from_json("bert_results.json")
print("Loaded:", loaded.model_name, "| SOD:", loaded.semantic_onset_depth)